In [ ]:
import os
import sys

# 将项目根目录加入 sys.path（根据实际情况修改路径）
project_root = os.path.abspath('..')   # 或 os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 然后绝对导入
from utils import Processor, Calculator

In [ ]:
import numpy as np
from pathlib import Path

In [ ]:
dates = np.load('/data/xujiayi/xjy/axis/dates.npy', allow_pickle=True)
ticks = np.load('/data/xujiayi/xjy/axis/ticks.npy', allow_pickle=True)

dclose = np.memmap('/data/xujiayi/xjy/d_field/close_adj.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
dopen = np.memmap('/data/xujiayi/xjy/d_field/open_adj.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
dhigh = np.memmap('/data/xujiayi/xjy/d_field/high_adj.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
dlow = np.memmap('/data/xujiayi/xjy/d_field/low_adj.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
dvolume = np.memmap('/data/xujiayi/xjy/d_field/volume_adj.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')
dturnover = np.memmap('/data/xujiayi/xjy/d_field/turnover.bin',dtype=float,shape=(len(dates),len(ticks)),mode='r')


In [ ]:
output_dir = Path('/data/xujiayi/xjy/research_factors/model_input/dGRU')

In [ ]:
close2open = np.divide(dclose, dopen, out=np.zeros_like(dclose), where=dopen!=0) -1
high2open = np.divide(dhigh, dopen, out=np.zeros_like(dhigh), where=dopen!=0) -1
low2open = np.divide(dlow, dopen, out=np.zeros_like(dlow), where=dopen!=0) -1
high2close = np.divide(dhigh, dclose, out=np.zeros_like(dhigh), where=dclose!=0) -1
low2close = np.divide(dlow, dclose, out=np.zeros_like(dlow), where=dclose!=0) -1
high2low = np.divide(dhigh, dlow, out=np.zeros_like(dhigh), where=dlow!=0) -1

In [ ]:
close_pct = Processor.winsorize_clip(Calculator.pct_change(dclose))
open_pct = Processor.winsorize_clip(Calculator.pct_change(dopen))
high_pct = Processor.winsorize_clip(Calculator.pct_change(dhigh))
low_pct = Processor.winsorize_clip(Calculator.pct_change(dlow))
logvolume_pct = Processor.winsorize_clip(Calculator.pct_change(np.log(dvolume)))
turnover_pct = Processor.winsorize_clip(Calculator.pct_change(dturnover))

In [ ]:
close_zscore = Processor.cross_standardize(dclose)
high_zscore = Processor.cross_standardize(dhigh)
low_zscore = Processor.cross_standardize(dlow)
open_zscore = Processor.cross_standardize(dopen)
logvolume_zscore = Processor.cross_standardize(np.log(dvolume))
turnover_zscore = Processor.cross_standardize(dturnover)

In [ ]:
d_fields = [
    'close_zscore','open_zscore','high_zscore','low_zscore','logvolume_zscore','turnover_zscore',
    'close_pct','open_pct','high_pct','low_pct','logvolume_pct','turnover_pct',
    'close2open','high2open','low2open','high2low','high2close','low2close',
]

In [ ]:
field_dict = {
    "close_zscore": close_zscore,
    "open_zscore": open_zscore,
    "high_zscore": high_zscore,
    "low_zscore": low_zscore,
    "logvolume_zscore": logvolume_zscore,
    "turnover_zscore": turnover_zscore,

    "close_pct": close_pct,
    "open_pct": open_pct,
    "high_pct": high_pct,
    "low_pct": low_pct,
    "logvolume_pct": logvolume_pct,
    "turnover_pct": turnover_pct,

    "close2open": close2open,
    "high2open": high2open,
    "low2open": low2open,
    "high2low": high2low,
    "high2close": high2close,
    "low2close": low2close,
}

for field in d_fields:
    var = field_dict[field]
    var.astype(float).tofile(output_dir/f'{field}.bin')